# 01 - Extraccion de trayectoria desde imagenes

Partimos desde frames sinteticos de un lanzamiento de proyectil. El objetivo es convertir pixeles en una tabla con tiempo y posicion fisica.


In [ ]:
from pathlib import Path
import csv, subprocess, sys
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
FRAME_DIR = ROOT / 'data' / 'frames'
OUT_CSV = ROOT / 'data' / 'trajectory_extracted.csv'
frames = sorted(FRAME_DIR.glob('frame_*.png'))
if not frames:
    subprocess.run([sys.executable, str(ROOT / 'scripts' / 'generate_assets.py')], check=True)
    frames = sorted(FRAME_DIR.glob('frame_*.png'))
print(f'Frames encontrados: {len(frames)}')
frames[:3]


In [ ]:
img = Image.open(frames[0]).convert('RGB')
plt.figure(figsize=(8,4))
plt.imshow(img)
plt.axis('off')
plt.show()


## Segmentacion por color

Como la pelota es roja y el fondo es claro, usamos una regla simple. En un video real esta etapa puede reemplazarse por seguimiento manual, OpenCV o un detector mas robusto.


In [ ]:
def detect_red_ball(path):
    img = Image.open(path).convert('RGB')
    arr = np.asarray(img).astype(float)
    r, g, b = arr[...,0], arr[...,1], arr[...,2]
    mask = (r > 150) & (r > 1.6*g) & (r > 1.6*b)
    yy, xx = np.nonzero(mask)
    if len(xx) == 0:
        return np.nan, np.nan, mask
    return xx.mean(), yy.mean(), mask

xpx, ypx, mask = detect_red_ball(frames[0])
plt.figure(figsize=(8,4))
plt.imshow(mask, cmap='gray')
plt.scatter([xpx], [ypx], c='cyan', s=60)
plt.axis('off')
plt.show()


In [ ]:
fps = 20.0
px_per_m = 140.0
origin_px = np.array([70.0, 540.0 - 58.0])
rows = []
for k, frame in enumerate(frames):
    xpx, ypx, _ = detect_red_ball(frame)
    t = k / fps
    x_m = (xpx - origin_px[0]) / px_per_m
    y_m = (origin_px[1] - ypx) / px_per_m
    rows.append((k, t, xpx, ypx, x_m, y_m))

with OUT_CSV.open('w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['frame','t_s','x_px','y_px','x_m','y_m'])
    writer.writerows(rows)
print(f'Datos guardados en {OUT_CSV}')


In [ ]:
data = np.array([[r[1], r[4], r[5]] for r in rows])
t, x, y = data[:,0], data[:,1], data[:,2]
plt.figure(figsize=(7,4))
plt.plot(x, y, 'o-', label='trayectoria extraida')
plt.xlabel('x [m]')
plt.ylabel('y [m]')
plt.gca().set_aspect('equal', adjustable='box')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


## Preguntas de cierre

- Que supuestos usamos para convertir pixeles a metros?
- Que podria fallar si la pelota no fuera roja?
- Que cambia si el video real tiene perspectiva, sombras o movimiento de camara?
